# Notebook 13 - Resultados (graficas) — Dynamic Augmented Adaptive Group Counting

Tres resultados del proyecto, cada uno como una grafica. Cada celda tiene sus
parametros al inicio: cambialos y vuelve a correr para explorar. Todo sale de
correr el codigo del paquete `augmented`.


In [ ]:
%matplotlib inline
import os, sys
_d = os.path.abspath(os.getcwd())
while _d != os.path.dirname(_d) and not os.path.isfile(os.path.join(_d, "augmented", "__init__.py")):
    _d = os.path.dirname(_d)
if not os.path.isfile(os.path.join(_d, "augmented", "__init__.py")):
    _fb = "/Users/hectorbecerrilvillamil/Desktop/GroupCounting/group-count-dynamic"
    if os.path.isfile(os.path.join(_fb, "augmented", "__init__.py")):
        _d = _fb
if _d not in sys.path:
    sys.path.insert(0, _d)
import numpy as np
import matplotlib.pyplot as plt
import random
random.seed(0); np.random.seed(0)
print("repo root:", _d)


## Conteo vs si/no

Cuanto gana saber el conteo exacto por pool en vez de solo si/no, segun la prevalencia.

In [ ]:
# PARAMETROS (cambialos y re-corre)
n = 5                                 # personas por instancia
B = 2                                 # numero de tests
G = 3                                 # tamano maximo de pool
n_instancias = 20                     # instancias por nivel de prevalencia
prevalencias = [0.1, 0.2, 0.3, 0.4, 0.5]
seed = 0
spread = 0.05                         # ancho de la Uniforme alrededor de la prevalencia

from augmented.solver import solve_optimal_dapts
from augmented.classical_solver import solve_classical_dynamic

rng = np.random.default_rng(seed)
util_vals = [1, 2, 3]

medias, errores = [], []
for prev in prevalencias:
    rel_advs = []
    for _ in range(n_instancias):
        p = np.clip(rng.uniform(prev - spread, prev + spread, size=n), 1e-3, 1 - 1e-3)
        p = [float(x) for x in p]
        u = [int(x) for x in rng.choice(util_vals, size=n)]
        U_aug, _ = solve_optimal_dapts(p, u, B, G)      # optimo AUGMENTED (conteo exacto)
        U_cls, _ = solve_classical_dynamic(p, u, B, G)  # optimo CLASICO (si/no)
        rel_advs.append(100.0 * (U_aug - U_cls) / U_cls if U_cls > 1e-9 else 0.0)
    rel_advs = np.array(rel_advs)
    medias.append(rel_advs.mean())
    errores.append(rel_advs.std(ddof=1) / np.sqrt(len(rel_advs)))

medias, errores = np.array(medias), np.array(errores)

plt.figure(figsize=(7, 4.5))
plt.errorbar(prevalencias, medias, yerr=errores, marker="o", capsize=4,
             color="#1f77b4", label="Ventaja relativa (augmented vs clasico)")
plt.fill_between(prevalencias, medias - errores, medias + errores, alpha=0.2, color="#1f77b4")
plt.axhline(0, color="gray", lw=0.8, ls="--")
plt.xlabel("Prevalencia media")
plt.ylabel("Ventaja relativa de utilidad (%)")
plt.title(f"Conteo exacto vs si/no (n={n}, B={B}, G={G}, {n_instancias} instancias)")
plt.legend()
plt.tight_layout()
plt.show()

U invertida: la ventaja pega fuerte en prevalencia media (~0.3) y casi desaparece en los extremos, donde el conteo ya no dice nada nuevo. Siempre conteo-no-cero (min 0.23%).

## Brecha greedy vs optimo crece con B

Greedy contra el optimo augmented exacto, subiendo el presupuesto B.

In [ ]:
from augmented.solver import solve_optimal_dapts
from augmented.greedy import greedy_myopic_counting_expected_utility

# ----------------- PARAMETROS (cambialos y re-corre) -----------------
n            = 5                 # personas por instancia (manten <=5 para que sea rapido)
G            = 3                 # tamano maximo de cada test
prev_low     = 0.10              # prevalencia minima (rango de p_i)
prev_high    = 0.40              # prevalencia maxima (rango de p_i)
n_instancias = 25               # instancias aleatorias por cada B
grilla_B     = [1, 2, 3, 4]      # presupuestos de test a comparar (B<=4)
seed         = 7                 # semilla de reproducibilidad
# ---------------------------------------------------------------------

rng = np.random.default_rng(seed)

# Genera el mismo conjunto de instancias para todos los B (comparacion limpia)
instancias = []
for _ in range(n_instancias):
    p = rng.uniform(prev_low, prev_high, size=n).tolist()
    u = rng.uniform(0.5, 1.5, size=n).tolist()
    instancias.append((p, u))

medias, errores = [], []
for B in grilla_B:
    gaps = []
    for p, u in instancias:
        U_opt, _ = solve_optimal_dapts(p, u, B, G)                     # optimo AUGMENTED exacto
        U_gre    = greedy_myopic_counting_expected_utility(p, u, B, G)  # mejor greedy
        if U_opt > 1e-9:
            gaps.append(100.0 * (U_opt - U_gre) / U_opt)               # brecha relativa en %
    gaps = np.array(gaps)
    medias.append(gaps.mean())
    errores.append(gaps.std(ddof=1) / np.sqrt(len(gaps)))              # error estandar de la media

medias, errores = np.array(medias), np.array(errores)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.errorbar(grilla_B, medias, yerr=errores, marker="o", capsize=4,
            color="#c0392b", ecolor="#c0392b", linewidth=2, label="brecha media +/- EEM")
ax.fill_between(grilla_B, medias - errores, medias + errores, color="#c0392b", alpha=0.15)
ax.set_xlabel("Presupuesto de tests B")
ax.set_ylabel("Brecha relativa (U_opt - U_greedy) / U_opt  [%]")
ax.set_title("La brecha greedy vs optimo augmented se abre conforme crece B")
ax.set_xticks(grilla_B)
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

Con B=1 empatan; mas tests y el greedy miope se queda atras (hasta ~12% en B=4) por no planear hacia adelante.

## Gibbs se atasca

Apago el atajo exacto y mido el error del MCMC. Un pool: mezcla bien. Dos pools solapados: atrapado.

In [ ]:
# --- IMPORTS (necesarios antes de usar mask_from_indices) ---
import augmented.bayesian as B
from augmented.bayesian import gibbs_update, bayesian_update_by_counting
from augmented.core import mask_from_indices

# --- PARAMETROS (cambia y re-corre) ---
SEED = 0
ITER_GRID = [100, 200, 500, 1000, 2000, 5000, 10000]  # eje x (escala log)

# Caso CONVERGE: un solo pool {0,1,2,3} con 2 activos, n=4.
n_conv = 4
p_conv = [0.3, 0.3, 0.3, 0.3]
hist_conv = ((mask_from_indices([0, 1, 2, 3]), 2),)

# Caso ATRAPADO: dos pools solapados (comparten al agente 2), n=5.
n_trap = 5
p_trap = [0.3, 0.3, 0.3, 0.3, 0.3]
hist_trap = ((mask_from_indices([0, 1, 2]), 1), (mask_from_indices([2, 3, 4]), 1))
# ---------------------------------------

# Desactivar el atajo exacto: forzar siempre el camino MCMC.
B.EXACT_ACTIVE_THRESHOLD = 0
B.EXACT_ACTIVE_FALLBACK_CAP = 0

# Verdad: posterior exacta por conteo sobre todos los mundos consistentes.
exact_conv = bayesian_update_by_counting(p_conv, hist_conv, n_conv)
exact_trap = bayesian_update_by_counting(p_trap, hist_trap, n_trap)

def max_err(gibbs, exact):
    return max(abs(g - e) for g, e in zip(gibbs, exact))

err_conv, err_trap = [], []
for it in ITER_GRID:
    bi = max(10, it // 10)  # burn_in pequeno proporcional
    g_conv = gibbs_update(p_conv, hist_conv, n_conv, num_iterations=it,
                          burn_in=bi, tolerance=0.0, seed=SEED)
    g_trap = gibbs_update(p_trap, hist_trap, n_trap, num_iterations=it,
                          burn_in=bi, tolerance=0.0, seed=SEED)
    err_conv.append(max_err(g_conv, exact_conv))
    err_trap.append(max_err(g_trap, exact_trap))

fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.plot(ITER_GRID, err_conv, "o-", color="#2a7", lw=2,
        label="Un pool: {0,1,2,3}=2 (n=4)")
ax.plot(ITER_GRID, err_trap, "s-", color="#c33", lw=2,
        label="Pools solapados: {0,1,2}=1, {2,3,4}=1 (n=5)")
ax.set_xscale("log")
ax.set_xlabel("Iteraciones de Gibbs")
ax.set_ylabel(r"Error maximo  $\max_i\,|q_i^{Gibbs}-q_i^{exacto}|$")
ax.set_title("Gibbs forzado (sin atajo exacto): converge vs. atrapado")
ax.axhline(0.0, color="gray", lw=0.8, ls=":")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
plt.show()

Un pool (verde) baja a ~0.015; pools solapados (rojo) se clavan en ~0.37. Con `{0,1,2}=1` y `{2,3,4}=1`, voltear solo al agente 2 rompe un conteo, asi que la cadena nunca cruza esa particion y reporta su marginal como 0 (el exacto es 0.368).